# 🧦 新手操作指南 · 从零到跑通智能袜子姿态识别系统

> **目标读者**：从没接触过这个项目的同学。按本文件顺序走完就能跑起来。
>
> **你最后看到什么**：穿上双脚压力袜 → 电脑 UI 实时显示 8 通道数据 → 自动识别  
> 「坐着 / 跨腿坐 / 站正 / 往左偏 / 往右偏 / 往前走 / 往后走 / 上楼 / 下楼」。

---

## 0. 本文件和另一本 notebook 的分工

| 文件 | 面向谁 | 主要内容 |
|------|--------|----------|
| **本文件 · `quickstart_guide.ipynb`** | 只想把系统跑起来 | 装环境 → 文件目录 → 一条条命令 → 常见问题 |
| `smart_sock_pipeline.ipynb` | 想深入算法 | 归一化 / 自适应 / 特征工程 / 四棵 RF / 级联 的完整推导 |

**建议路径**：先用本指南把东西跑通，遇到「为什么这么做」时再翻 `smart_sock_pipeline.ipynb`。

## 第一步 · 装好 Python 环境

1. **装 Python 3.10 或 3.11 或 3.12**（本项目实测 Python 3.12.7 ✅）  
   官网：<https://www.python.org/downloads/>  
   Windows 下安装时记得勾 `Add python.exe to PATH`。

2. **打开 PowerShell / 终端**，`cd` 进项目根目录：
   ```powershell
   cd "C:\Users\<你自己的用户名>\Desktop\gaokao-university-database"
   ```

3. **一条命令装齐全部依赖**：
   ```bash
   pip install -r requirements.txt
   ```

   requirements.txt 里只有最小集：`numpy`、`scipy`、`PyQt5`、`scikit-learn`、`joblib`。如果你还想运行本文件里的代码 cell 和 `smart_sock_pipeline.ipynb`，再加一个：
   ```bash
   pip install matplotlib jupyter
   ```

4. **自检**：运行下面这个 cell，如果每个模块都能 import 就说明环境 OK。

In [1]:
import sys, platform
print('Python     :', sys.version.split()[0], '  OS:', platform.system(), platform.release())

ok = True
for mod in ('numpy', 'scipy', 'sklearn', 'joblib', 'PyQt5.QtCore'):
    try:
        __import__(mod)
        print(f'  ✅ {mod}')
    except Exception as e:
        print(f'  ❌ {mod}  -- {e}')
        ok = False

# 项目内模块
for mod in ('adaptive_preprocessing', 'ml_activity_features',
            'ml_branch_models', 'ml_train_branch_rfs',
            'personal_calibration', 'realtime_recognizer',
            'foot_pressure_monitor'):
    try:
        __import__(mod)
        print(f'  ✅ {mod} (本项目)')
    except Exception as e:
        print(f'  ❌ {mod} (本项目)  -- {e}')
        ok = False

print('\n整体状态：', 'OK 可以继续' if ok else '有缺件，请回到上一步装依赖')

Python     : 3.12.7   OS: Windows 11
  ✅ numpy
  ✅ scipy
  ✅ sklearn
  ✅ joblib
  ✅ PyQt5.QtCore
  ✅ adaptive_preprocessing (本项目)
  ✅ ml_activity_features (本项目)
  ✅ ml_branch_models (本项目)
  ✅ ml_train_branch_rfs (本项目)
  ✅ personal_calibration (本项目)
  ✅ realtime_recognizer (本项目)
  ✅ foot_pressure_monitor (本项目)

整体状态： OK 可以继续


## 第二步 · 认识一下项目目录

```
gaokao-university-database/             ← 项目根目录
├── foot_pressure_monitor.py            ← ★ UI 程序（跑起来就是这个）
├── realtime_recognizer.py              ← 识别器内核：级联 + 4 只 RF 推理
├── ml_train_branch_rfs.py              ← ★ 训练脚本：一条命令训出 4 只 RF
├── ml_activity_features.py             ← 特征工程 + 滑窗
├── ml_branch_models.py                 ← RF 4 只的封装
├── adaptive_preprocessing.py           ← 在线 EWMA + 冻结模式
├── personal_calibration.py             ← 个人量程 + 全局统计归一化
│
├── personal_calibration.json           ← 训练完成后产出的标定参数（UI 热重载也读它）
├── rf_active_motion.joblib             ← 4 只 RF 模型（训练后出现）
├── rf_active_static.joblib
├── rf_inactive_motion.joblib
├── rf_inactive_static.joblib
│
├── saving_data/                        ← ★ 所有标注 CSV 放这里
│   ├── sensor_data_dual_labeled_2026xxxx.csv   (33 份，训练用)
│   └── sensor_data_dual_raw_2026xxxx.csv       (采集时一起存的原始流)
│
├── smart_sock_pipeline.ipynb           ← 深度教学 notebook
├── quickstart_guide.ipynb              ← ★ 本文件
├── requirements.txt
└── README.md
```

**★ 的四个文件就是你主要会打交道的**。 其余都是被它们调用的内部逻辑。

## 第三步 · 确认 `saving_data/` 里有 33 份带标签 CSV

训练脚本只认 `sensor_data_dual_labeled_*.csv` 命名的文件。项目自带 33 份（两脚同时采集）。下面这段代码点一下就知道现场情况。

In [2]:
import os, glob
import ml_activity_features as maf

pattern = os.path.join(maf.DATA_DIR, 'sensor_data_dual_labeled_*.csv')
files = sorted(glob.glob(pattern))
print(f'saving_data/ 目录下找到 {len(files)} 份双脚标注 CSV：')
for p in files[:6]:
    print('  ', os.path.basename(p))
if len(files) > 6:
    print(f'  ...  （省略中间 {len(files) - 6} 份）')
    
# 快速读一下总帧数 / 总标签分布
raw, labels, subjects, mode = maf.load_csv_files(maf.DATA_DIR, labeled_only=True, raw_adc=True)
print(f'\n总帧数：{raw.shape[0]}  ≈ {raw.shape[0]/maf.SAMPLE_HZ:.0f} 秒  '
      f'({raw.shape[0]/maf.SAMPLE_HZ/60:.1f} 分钟真实数据)')
print(f'通道数：{raw.shape[1]}  (顺序：L_Toe, L_Forefoot, L_Heel, L_Knee, R_Toe, R_Forefoot, R_Heel, R_Knee)')

from collections import Counter
print('\n标签分布：')
for lab, n in Counter(labels).most_common():
    print(f'  {lab:25s}  {n:>5d}  ({n/len(labels)*100:5.1f}%)')

print(f'\n📌 训练的所有归一化参数都会在这 {len(files)} 份 CSV 拼起来的整体上算，')
print('   不会从单个文件或前 N 行局部估计 —— 这是保证模型通用性的关键。')

saving_data/ 目录下找到 33 份双脚标注 CSV：
   sensor_data_dual_labeled_20260410_201914.csv
   sensor_data_dual_labeled_20260410_202137.csv
   sensor_data_dual_labeled_20260410_202318.csv
   sensor_data_dual_labeled_20260410_202757.csv
   sensor_data_dual_labeled_20260411_134628.csv
   sensor_data_dual_labeled_20260411_134922.csv
  ...  （省略中间 27 份）

总帧数：20308  ≈ 2031 秒  (33.8 分钟真实数据)
通道数：8  (顺序：L_Toe, L_Forefoot, L_Heel, L_Knee, R_Toe, R_Forefoot, R_Heel, R_Knee)

标签分布：
  SITTING_CROSSLEGGED         3624  ( 17.8%)
  WALKING_FORWARD             3369  ( 16.6%)
  WALKING_BACKWARD            2764  ( 13.6%)
  STANDING_LEFT_LEAN          2335  ( 11.5%)
  STANDING_RIGHT_LEAN         1735  (  8.5%)
  STANDING_UPRIGHT            1722  (  8.5%)
  SITTING_NORMAL              1696  (  8.4%)
  STAIRS_DOWN                 1514  (  7.5%)
  STAIRS_UP                   1405  (  6.9%)
  SIT_TO_STAND                 144  (  0.7%)

📌 训练的所有归一化参数都会在这 33 份 CSV 拼起来的整体上算，
   不会从单个文件或前 N 行局部估计 —— 这是保证模型通用性的关键。


## 第四步 · 一键训练 4 只随机森林（含 val / test / 混淆矩阵）

**一条命令搞定**。它会在后台做完整流水线：

1. 把 `saving_data/` 里 **全部 33 份 CSV** 拼成一个整体矩阵；
2. 在整体上算 **5 个全局统计**（`baseline_raw / press_min / press_max / press_mean / press_std`）；
3. 把这些写进 `personal_calibration.json`；
4. 冻结自适应预处理（EWMA 关掉，用全局量归一化每一帧）；
5. 滑窗（10 帧 / 步 2 帧）+ 特征工程；
6. 按标签分桶 → 训 4 只独立的 RandomForest；
7. **三段切分评估**：每只 RF 都会输出 CV 交叉验证 / validation / test 三套准确率；
8. **混淆矩阵**：每只 RF 的 val 集和 test 集各一张 PNG，落盘到 `confusion_matrices/`；
9. 导出 4 个 `rf_*.joblib`。

### 没有额外数据集怎么办？

用 **3-way split**：从 33 份 CSV 做出来的窗口里按 `60% / 20% / 20%` **stratify** 切成 train / validation / test —— 三段**互不重叠**。  
再在 **train** 上做 **5-Fold Stratified CV**（不碰 val/test），得到 val-accuracy 的 `mean ± std`，比单次 val 更稳。

### 命令

```bash
python ml_train_branch_rfs.py                      # 默认 60/20/20
python ml_train_branch_rfs.py --plots-dir ""       # 不要 PNG（跑得更快）
python ml_train_branch_rfs.py --calib personal_calibration.json   # 用已有标定
```

⚙️ 在本 notebook 内也可以跑：下一格就是。首次训练大约 **1 ~ 2 分钟**（33 份 CSV × 4 棵树 + 每棵 CV 5 折）。

In [3]:
import importlib, ml_train_branch_rfs as mtb, ml_branch_models as mbm
importlib.reload(mtb)

exit_code = mtb.main(argv=[])   # 在 notebook 里调用必须显式传 argv=[]，否则会读到 kernel 自己的 -f 参数
print('\n训练脚本返回码:', exit_code, '(0 代表成功)')
print('\n当前目录下应该看到这 4 个模型：')
for fn in mbm.BRANCH_TO_FILE.values():
    size = os.path.getsize(fn) / 1024 if os.path.isfile(fn) else 0
    print(f'  {fn:34s}  {"OK" if size > 0 else "缺失"}  ({size:.1f} KB)')

print('\n以及一份标定 JSON：')
print('  personal_calibration.json  OK' if os.path.isfile('personal_calibration.json') else '  缺失！')

[calib] auto-fit offline calibration saved → C:\Users\Collin EP\Desktop\gaokao-university-database\personal_calibration.json
PersonalCalibration(subject='offline_auto_population', source='offline_auto_csv')
  channel         min_raw   max_raw   span
  L_Toe              0.0    4095.0    4095.0
  L_Forefoot      1792.0    4095.0    2303.0
  L_Heel          1088.4    4095.0    3006.6
  L_Knee            31.0    4095.0    4064.0
  R_Toe           2390.4    4095.0    1704.6
  R_Forefoot      2132.0    4095.0    1963.0
  R_Heel          2145.4    4095.0    1949.6
  R_Knee           179.0    4095.0    3916.0

  GLOBAL STATS (used to freeze EWMA baseline / range / z-score):
  channel         baseline   p_min     p_max     p_mean    p_std
  L_Toe           4095.0      0.0   4095.0   2128.0   1499.6
  L_Forefoot      4095.0      0.0   2258.0    824.2    716.9
  L_Heel          4095.0      0.0   3012.0   1683.4   1161.1
  L_Knee          4095.0      0.0   4047.0    961.1   1565.8
  R_Toe        

### 4.1 · 看看 4 只 RF 的混淆矩阵（val & test）

训练脚本生成了 8 张 PNG 到 `confusion_matrices/` 目录。下面这格把它们全部嵌入显示。

In [ ]:
from IPython.display import Image, display, Markdown

BRANCHES = ['active_motion', 'active_static', 'inactive_motion', 'inactive_static']
cm_dir = 'confusion_matrices'

for br in BRANCHES:
    display(Markdown(f'#### · Branch = `{br}`'))
    row = []
    for split in ('val', 'test'):
        p = os.path.join(cm_dir, f'confusion_{br}_{split}.png')
        if os.path.isfile(p):
            row.append(p)
        else:
            print(f'(缺失: {p})')
    # 并排显示 val / test
    for p in row:
        display(Image(filename=p, width=480))
    print()

### 4.2 · 只看分数表（不用重训）

训练脚本把每只 RF 的 CV / val / test 分数都写到了 `rf_*.joblib` 的 `metrics` 字段里。下面这格直接读出来做一张表。

In [ ]:
import joblib

print(f'{"branch":<18}{"N":>7}{"CV mean":>10}{"CV std":>10}{"val":>9}{"test":>9}')
print('-' * 63)
for br, fn in mbm.BRANCH_TO_FILE.items():
    if not os.path.isfile(fn):
        print(f'{br:<18}  (no model file)')
        continue
    bundle = joblib.load(fn)
    m = bundle.get('metrics', {})
    if not m:
        print(f'{br:<18}  (model has no metrics — retrain to populate)')
        continue
    print(f'{br:<18}{m["n_samples"]:>7}{m.get("cv_mean", float("nan")):>10.4f}'
          f'{m.get("cv_std", float("nan")):>10.4f}'
          f'{m["val_accuracy"]:>9.4f}{m["test_accuracy"]:>9.4f}')

## 第五步 · 把训练出来的模型在离线 CSV 上复盘

训练完马上在同一份 33 CSV 上跑一遍识别器，确认：

- Layer 1（膝盖门控）按规则切 `ACTIVE` / `INACTIVE`；
- Layer 2（运动 / 静态）正确激活 4 只 RF 之一；
- 最终 state 跟 CSV 的 ground-truth Label 基本对得上（达标的 accuracy 应在 95% 以上，实测 98–100%）。

In [ ]:
import realtime_recognizer as rrz
from collections import Counter

rec = rrz.OnlineRecognizer(calibration='personal_calibration.json')
print(f'识别器 ready：bank 冻结={all(c._freeze for c in rec._adapt_bank.channels)}')
print(f'            加载了全局标定：{rec._calibration.has_global_stats}')

# 用上面读好的 raw / labels 一帧一帧喂
pred, gt = [], []
for i in range(raw.shape[0]):
    lt, lf, lh, lk, rt, rf, rh, rk = raw[i]
    out = rec.update_bilateral((lt, lf, lh, lk), (rt, rf, rh, rk), t=i / maf.SAMPLE_HZ)
    pred.append(out['state'])
    gt.append(labels[i])

# 简易 accuracy：只看"有定论"的帧（UNKNOWN 标签排除）
valid = [(p, g) for p, g in zip(pred, gt) if g not in ('UNKNOWN', 'SIT_TO_STAND')]
match = sum(1 for p, g in valid if p == g)
print(f'\n离线复盘 accuracy = {match / max(1, len(valid)):.1%}  '
      f'（{match} / {len(valid)} 帧，不含 UNKNOWN / SIT_TO_STAND）')
print('\n预测 state 分布（前 10）：')
for s, n in Counter(pred).most_common(10):
    print(f'  {s:25s}  {n}')

识别器 ready：bank 冻结=True
            加载了全局标定：True


## 第六步 · 启动实时 UI

> ⚠️ **这一步不能在 notebook 里跑**，PyQt UI 需要独立进程、独立事件循环。请在**项目根目录**打开一个新的 PowerShell / 终端：

```bash
python foot_pressure_monitor.py
```

### UI 窗口里每个按钮是干啥的

| 控件 | 说明 |
|------|------|
| `Left IP / Left Port` | 左脚 MCU 的 IP（默认 `0.0.0.0:5000`，表示本机监听 5000） |
| `Right IP / Right Port` | 右脚 MCU 的 IP（默认 `0.0.0.0:6000`） |
| `Start` / `Stop` | 打开 / 关闭两路 TCP 监听 |
| 状态灯 (绿/红) | 两只袜子的连接状态 |
| `Data capture` 选项卡 | 选标签 → Start → 系统会把原始和带标签的 CSV 写进 `saving_data/` |
| `Calibration` 选项卡 | 个人在线两步标定（见第七步） |

**识别结果实时显示在右上角 HUD**：Layer-1 / Layer-2 / 最终 state + RF 概率 + 步数。

### MCU 怎么连？

两只袜子（MCU 模块）要在同一个局域网内主动连到你电脑的 5000 和 6000 端口，**用 `\n` 分隔的 JSON**发送，例如：

```
{"toe":3800,"forefoot":3500,"heel":2900,"knee":4095,"ts":1718000000.1}
```

字段名大小写不敏感。实际采集率 **10 Hz**（每 100 ms 一帧）。

## 第七步 · 为新用户做一次【个人在线标定】（2 分钟）

每次换新测试者，最好跑一次个人标定。体重、脚型、绑带松紧都会影响压力读数。

### 操作步骤（全部在 UI 里点）

1. 打开 UI（第六步），两只袜子都**绿灯**。
2. 切到 **`Calibration`** 选项卡。
3. `Subject` 填这位测试者的代号（例如 `alice_20260417`）。
4. **第 1 步：自然直立站立**
   - 测试者站稳保持 5 秒
   - 点 `Start Step 1`，进度条到底 → 点 `Finish Step 1`
   - 背后程序取每个压力通道的 **1% 分位点** 作为"个人满载"值
5. **第 2 步：膝盖弯到约 90°**
   - 测试者蹲坐或坐下弯膝保持 5 秒
   - 点 `Start Step 2`，进度条到底 → 点 `Finish Step 2`
   - 背后取每个膝盖通道的 **1% 分位点** 作为"最大弯曲"值
6. 预览框会显示 8 通道的 `min / max / span` 表，**span 合理**（pressure ≥ 500，knee ≥ 1500）→ 点 `Save personal_calibration.json + reload`。
7. 主窗口**立即热重载**识别器，新的个人量程马上生效，无需重启 UI。

### ⚠️ 重要：换了标定后 4 只 RF 要不要重训？

- **不重训也能用** —— `PersonalCalibration.normalize_to_adc` 只做线性重映射，大多数常见姿态还是能识别。
- **想让 4 只 RF 真的"认识"这位测试者**，就再跑一次：
  ```bash
  python ml_train_branch_rfs.py --calib personal_calibration.json
  ```
  这会用新的个人标定参数重新抽特征 + 训树。**30 秒 ~ 1 分钟**，一劳永逸。

## 第八步 · 采集你自己的数据（可选）

如果你想扩充数据集、让模型学新的姿态，走这套流程：

1. UI 连上两只袜子（绿灯）。
2. 切到 `Data capture` 选项卡。
3. 点一个**标签按钮**（`STANDING_UPRIGHT` / `WALKING_FORWARD` / ... ）。
4. 点 **`Start`** → 系统开始写两份 CSV：
   - `saving_data/sensor_data_dual_raw_2026xxxx.csv` —— 原始 ADC 流
   - `saving_data/sensor_data_dual_labeled_2026xxxx.csv` —— 带你选中的标签
5. 做相应的动作（20–30 秒即可）。
6. 点 **`Stop`**，CSV 落盘。
7. 想换动作：点另一个标签 → `Start` → 再采一段。

### 推荐的标签清单（和已有 33 份 CSV 保持一致）

```
STANDING_UPRIGHT        站正
STANDING_LEFT_LEAN      身体往左偏
STANDING_RIGHT_LEAN     身体往右偏
SITTING_NORMAL          正常坐
SITTING_CROSSLEGGED     跨腿坐
WALKING_FORWARD         往前走
WALKING_BACKWARD        往后走
STAIRS_UP               上楼梯
STAIRS_DOWN             下楼梯
```

采完新数据，**直接重跑 `python ml_train_branch_rfs.py`**，新窗口会自动纳入训练。

## 第九步 · 常见问题 FAQ

### ❓ `python foot_pressure_monitor.py` 启动后没有绿灯
- 确认 MCU 和 PC 在同一网段；
- 确认电脑防火墙放行 5000 / 6000 端口；
- 换个 IP 手动填进 `Left IP / Right IP` 输入框再点 Start。

### ❓ UI 开起来但识别一直是 `STANDING_UPRIGHT` 不换
- 99% 是**标定没做 / 没用该测试者的标定**。做第七步的 2 分钟在线标定。
- 还不行就看终端输出里的 `knee raw = ...`，如果膝盖通道常年 `> 3500`，说明袜子没绑紧 / 传感器没弯。

### ❓ 识别误报 `WALKING_FORWARD`
- 项目里走路切换有**强约束**：必须真实抬脚 + 膝盖摆到 4080 以上。如果仍然误报：
  - 检查 `realtime_recognizer.py` 的 `WALK_ENTER_MIN_STEPS`、`WALK_KNEE_EXTEND_RAIL_TH`。
  - 提高 `MOTION_MIN_AMPLITUDE` 增强门槛。

### ❓ 想改滑动窗口长度 / 步长
- 改 `ml_activity_features.py` 顶部的 `ML_WINDOW_DURATION_S`（默认 1.0 s = 10 帧）和 `WINDOW_STEP`（默认 2 帧）。
- 改完**必须重训**：`python ml_train_branch_rfs.py`。

### ❓ 训练脚本报 `saving_data not found`
- 你当前工作目录不对。`cd` 回项目根目录再跑。

### ❓ 想完全清零重来
```bash
# 删掉训练产物，其余保留
rm rf_*.joblib personal_calibration.json
# 然后重跑
python ml_train_branch_rfs.py
```

## 你现在已经会了

跑通这 9 步，你就已经完成了：

- ✅ 装环境
- ✅ 看懂目录结构
- ✅ 确认 33 份训练数据
- ✅ 一键训练 4 只 RF（全局归一化）
- ✅ 离线在全集上复盘识别精度
- ✅ 启动实时 UI
- ✅ 为新用户做 2 分钟在线标定
- ✅ 采集你自己的新数据
- ✅ 排障 FAQ

> **想继续深入？** → 打开 `smart_sock_pipeline.ipynb`，那里每一个单元格都解释了**为什么这么设计**：
> 2 级级联为什么先 knee-gate、为什么走路要强约束、为什么要冻结 EWMA、4 只 RF 各自的特征维度和分类标签空间。

祝你玩得开心！